# 真实 API：P(True) 自评置信度

用同一模型的两种估计器评估自报置信度：Chat Completions logprob 路径读取 ` A`/` B` token 概率；Responses API 只能走 verbalized 回退。

## 真实集成前提

安装：`python3 -m pip install -e ".[openai]"`；logprob 需要服务端支持 `logprobs=True`（OpenAI-compatible 本地服务也适用）。

代码单元没有预执行。填写 API 凭据或本地模型绝对路径后，从上到下运行；不要将 API key 保存到 notebook。

## 参数说明

- `OPENAI_MODEL: str`：模型名。
- `use_samples_as_brainstorm: bool = True`：用采样作为 brainstorm 答案。
- `max_brainstorm: int = 4`：brainstorm 答案数上限。
- `normalize_ab: bool = True`：True/False 概率归一化；False 复现官方“只读 A”估计器。

## 步骤 1：配置真实后端

In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd()
if not (repo_root / "src" / "hallucination").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

import os
from getpass import getpass
from openai import OpenAI
from hallucination.detection.backends.api import OpenAIChatBackend, OpenAIResponsesBackend
OPENAI_MODEL = "gpt-4.1-mini"
TEMPERATURE = 0.8
api_key = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")
base_url = os.environ.get("OPENAI_BASE_URL")
client = OpenAI(api_key=api_key, **({"base_url": base_url} if base_url else {}))
chat_backend = OpenAIChatBackend(OPENAI_MODEL, client=client)
responses_backend = OpenAIResponsesBackend(OPENAI_MODEL, client=client, temperature=TEMPERATURE)

## 步骤 2：构建两种估计器

In [ ]:
from hallucination.detection import DetectionExample
from hallucination.detection.whitebox.p_true import PTrueScorer
logprob_scorer = PTrueScorer(chat_backend, use_samples_as_brainstorm=True, max_brainstorm=4, normalize_ab=True)
verbalized_scorer = PTrueScorer(responses_backend)

## 步骤 3：交互运行并分别读取

In [ ]:
prompt = input("Prompt：").strip()
samples = responses_backend.sample(prompt, 5)
example = DetectionExample(prompt=prompt, response=samples[0], samples=samples[1:])
for name, scorer in (("logprob", logprob_scorer), ("verbalized", verbalized_scorer)):
    result = scorer.score(example)
    print(name, "| method:", result.raw["method"], "| score:", result.score, "| risk:", result.risk)
    print("  details:", result.raw.get("token_details") or result.raw.get("fallback_reason"))

## 结果解读与限制

P(True) 是自我报告，普遍存在过度自信；两种估计器都未校准，比较时必须固定估计器类型。